**<h1>Electric Line Extension - Analysis 6**
#### SDG&E 2023 Annual Report - Cleaning
<i>Any question regarding the notebook, please contact Robert Ford<br>
    Last Updated: 07/06/2026 | Start Development: 07/06/2026</i>
* Source: SDG&E Annual Report 2023 (`data/raw/SDG&E_ Annual Report_2023.xlsx`)
* Goal: Clean the messy annual-report layout (4 sheets, 3-row header, repeating month/customer-class
  blocks) into the same long/tidy format used for the rest of the master line-extension dataset
  (Month, Year, Customer Class, Type, Count, Fuel_Structure_Type, IoU, Climate Zone, Multi Dwelling),
  then append it to the master spreadsheet.


In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import os


**Clean 6 — SDG&E 2023 Annual Report**

The raw file has 4 sheets (one per Fuel_Structure_Type): `All Electric New Construction`,
`Mixed-Fuel New Construction`, `Mixed-Fuel Upgrades`, `All Electric Upgrades`.

Each sheet has a 3-row title/notes header, then repeats this block 12 times (once per month):
1. A single date value in column A (e.g. `2023-01-01`) marking the start of that month's block
2. A header row repeating `Customer Class` in column A and the 11 Type names in columns B–L
3. Four data rows, one per Customer Class (`Residential`, `Industrial`, `Commercial`, `Agriculture`)

Below the last month there are summary/footnote rows which we ignore.

The parser below walks each sheet row-by-row: it tracks the current month whenever it sees a date
in column A, tracks the current set of Type column names whenever it sees a `Customer Class` header
row, and emits one long-format record per (month, customer class, type) whenever it sees a data row.


In [ ]:
# ============================================
# SET YOUR INPUT / OUTPUT PATHS
# ============================================

raw_path = (
    r"C:/Users/RFord/OneDrive - California Energy Commission"
    r"/Documents/Analysis - Scripts and Code"
    r"/Electric-Line-Extension-Data/data/raw"
    r"/SDG&E_ Annual Report_2023.xlsx"
)

processed_folder = (
    r"C:/Users/RFord/OneDrive - California Energy Commission"
    r"/Documents/Analysis - Scripts and Code"
    r"/Electric-Line-Extension-Data/data/processed"
)

os.makedirs(processed_folder, exist_ok=True)


In [ ]:
VALID_CLASSES = {"Residential", "Industrial", "Commercial", "Agriculture"}

# Map raw sheet names -> standardized Fuel_Structure_Type values
SHEET_TO_FUEL_STRUCTURE = {
    "All Electric New Construction": "All Electric New Construction",
    "Mixed-Fuel New Construction": "Mixed Fuel New Construction",
    "Mixed-Fuel Upgrades": "Mixed Fuel Upgrades",
    "All Electric Upgrades": "All Electric Upgrades",
}


def parse_sheet(ws, fuel_structure_type):
    """Walk a single sheet row-by-row, pulling the repeating
    month-block / customer-class-block structure into long records."""
    records = []
    current_month = None
    type_columns = None  # [(col_index, type_name), ...] from the most recent header row

    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, max_col=12):
        first_cell = row[0].value

        # A) Month marker row: a datetime sitting alone in column A
        if hasattr(first_cell, "strftime"):
            current_month = first_cell.strftime("%b")
            continue

        # B) Header row: "Customer Class" in column A -> (re)capture Type names from B:L
        if isinstance(first_cell, str) and first_cell.strip() == "Customer Class":
            type_columns = [(i, cell.value) for i, cell in enumerate(row[1:], start=2) if cell.value]
            continue

        # C) Data row: one of the 4 known Customer Class values in column A
        if isinstance(first_cell, str) and first_cell.strip() in VALID_CLASSES and current_month and type_columns:
            customer_class = first_cell.strip()
            for col_idx, type_name in type_columns:
                raw_val = row[col_idx - 1].value
                # Normalize "-" / blank placeholders to 0
                if raw_val is None or (isinstance(raw_val, str) and raw_val.strip() in ("-", "")):
                    count = 0
                else:
                    count = raw_val
                records.append({
                    "Month": current_month,
                    "Year": 2023,
                    "Customer Class": customer_class,
                    "Type": type_name.strip() if isinstance(type_name, str) else type_name,
                    "Count": count,
                    "Fuel_Structure_Type": fuel_structure_type,
                    "IoU": "SDG&E",
                    "Climate Zone": "Unknown",
                    "Multi Dwelling": "Unknown",
                })
        # D) Anything else (titles, footnotes, blank rows) is skipped
    return records


In [ ]:
wb = openpyxl.load_workbook(raw_path, data_only=True)

all_records = []
for sheet_name, fuel_structure_type in SHEET_TO_FUEL_STRUCTURE.items():
    ws = wb[sheet_name]
    recs = parse_sheet(ws, fuel_structure_type)
    print(f"{sheet_name}: {len(recs)} rows extracted")
    all_records.extend(recs)

sdge_2023 = pd.DataFrame(all_records, columns=[
    "Month", "Year", "Customer Class", "Type", "Count",
    "Fuel_Structure_Type", "IoU", "Climate Zone", "Multi Dwelling",
])

print("\nTotal rows:", len(sdge_2023))
sdge_2023.head(15)


**Sanity checks** — expect 2,112 rows total (4 sheets × 12 months × 4 customer classes × 11 types),
176 rows per month, 528 rows per Customer Class, 528 rows per Fuel_Structure_Type.


In [ ]:
print(sdge_2023["Month"].value_counts())
print()
print(sdge_2023["Customer Class"].value_counts())
print()
print(sdge_2023["Fuel_Structure_Type"].value_counts())
print()
print("Type values:")
for t in sdge_2023["Type"].unique():
    print(" ", t)


**Export** — save the standalone cleaned SDG&E 2023 file, then append it to the master spreadsheet.

Update `master_path` below to point at your actual master workbook/sheet before running this last cell.


In [ ]:
# Save the standalone cleaned file
sdge_2023_out_path = os.path.join(processed_folder, "SDGE_2023_Annual_Cleaned.xlsx")
sdge_2023.to_excel(sdge_2023_out_path, index=False)
print(f"Saved: {sdge_2023_out_path}")


In [ ]:
# ============================================
# Append to the master spreadsheet
# ============================================
# Update this path to your actual master file
master_path = os.path.join(processed_folder, "Master_Line_Extension_Data.xlsx")

if os.path.exists(master_path):
    master_df = pd.read_excel(master_path)
    master_df = pd.concat([master_df, sdge_2023], ignore_index=True)
    master_df.to_excel(master_path, index=False)
    print(f"Appended {len(sdge_2023)} rows to master file: {master_path}")
    print(f"New master shape: {master_df.shape}")
else:
    print(f"Master file not found at: {master_path}")
    print("Skipping append -- update 'master_path' above and re-run this cell,")
    print(f"or manually combine it with '{sdge_2023_out_path}'.")
